<a href="https://colab.research.google.com/github/GabrielLobaton18/Pred_Com_De_Casa/blob/main/Predicci%C3%B3n_de_compra_de_casas_CGML_Redes_Neuronales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Comenzamos con poniendo las configuraciones iniciales necesarias

In [1]:
#como siempre instalamos lo usual, lo que seguramente necesitaremos
%pip install -q tensorflow
%pip install -q mlflow
%pip install -q pandas
%pip install -q matplotlib
%pip install -q scikit-learn
%pip install -q kaggle
%pip install -q dagshub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 756.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
#reciclamos las importaciones del código pásado por si algo hace falta
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import mlflow
import dagshub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
import zipfile
import shutil
from google.colab import files, drive

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [3]:
dagshub.init(repo_owner='GabrielLobaton18', repo_name='Pred_Com_De_Casa', mlflow=True)
with mlflow.start_run():
  # Your training code here...
  mlflow.log_metric('accuracy', 42)
  mlflow.log_param('Param name', 'Value')

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=a2e82dde-1fd1-46ab-9d2a-2af02612bdce&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=f3c591b074a7224f934a64adf35a61f063a54239ffdc5443ee96fa363a60c509




Accessing as GabrielLobaton18

Initialized MLflow to track repo "GabrielLobaton18/Pred_Com_De_Casa"

Repository GabrielLobaton18/Pred_Com_De_Casa initialized!

🏃 View run enchanting-wasp-424 at: https://dagshub.com/GabrielLobaton18/Pred_Com_De_Casa.mlflow/#/experiments/0/runs/d4ceed0ce89142d990b0ecf58585ec99
🧪 View experiment at: https://dagshub.com/GabrielLobaton18/Pred_Com_De_Casa.mlflow/#/experiments/0


Descargamos

In [4]:
# Configurar las variables de entorno de Kaggle
os.environ['KAGGLE_USERNAME'] = 'Gabriel Lobatón'
os.environ['KAGGLE_KEY'] = 'KGAT_da409035364bf1d526a25d9cf4e6e310'

In [7]:
# Descargamos el dataset de compra de casas
!kaggle datasets download -d mohankrishnathalla/global-house-purchase-decision-dataset

# Descomprimimos en una carpeta específica
!unzip -q global-house-purchase-decision-dataset.zip -d /content/house_dataset

# Verificamos la descarga
!ls /content/house_dataset

Dataset URL: https://www.kaggle.com/datasets/mohankrishnathalla/global-house-purchase-decision-dataset
License(s): CC0-1.0
global-house-purchase-decision-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
replace /content/house_dataset/global_house_purchase_dataset.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
global_house_purchase_dataset.csv


In [10]:
# Cargamos el archivo de atributos
attr_df = pd.read_csv('/content/global-house-purchase-decision-dataset.zip')

# Mostramos las primeras filas
print(attr_df.head())

# Visualizamos la forma del dataframe
print("Dimensiones:", attr_df.shape)

   property_id       country          city property_type furnishing_status  \
0            1        France     Marseille     Farmhouse    Semi-Furnished   
1            2  South Africa     Cape Town     Apartment    Semi-Furnished   
2            3  South Africa  Johannesburg     Farmhouse    Semi-Furnished   
3            4       Germany     Frankfurt     Farmhouse    Semi-Furnished   
4            5  South Africa  Johannesburg     Townhouse   Fully-Furnished   

   property_size_sqft    price  constructed_year  previous_owners  rooms  ...  \
0                 991   412935              1989                6      6  ...   
1                1244   224538              1990                4      8  ...   
2                4152   745104              2019                5      2  ...   
3                3714  1110959              2008                1      3  ...   
4                 531    99041              2007                6      3  ...   

   customer_salary  loan_amount  loan_tenure

ahora pasaremos el texto como variable categorica y normalizaremos la variables númericas